# Manna 11–17 to Layperson boundary audit

Reproducible checks for the complete 8,741-question boundary analysis. This notebook uses only Python's standard library and reads the generated question-level outputs; it does not contain a second copy of the allocator.

In [1]:
from pathlib import Path
import csv, json, statistics

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'reports/audience_reallocation/summary.json').exists():
    ROOT = ROOT.parent
REPORT = ROOT / 'reports/audience_reallocation'
summary = json.loads((REPORT / 'summary.json').read_text(encoding='utf-8'))
with (REPORT / 'all_questions.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
with (REPORT / 'changes.csv').open(encoding='utf-8', newline='') as handle:
    changes = list(csv.DictReader(handle))
print(f"Loaded {len(rows):,} boundary questions and {len(changes):,} changes.")


Loaded 8,741 boundary questions and 3,822 changes.


## Before and after distribution

In [2]:
print('Stage  Audience    Count  Readiness median  P25  P75  Adult-content median')
for stage in ('before', 'after'):
    for item in summary[stage]:
        print(f"{stage.title():<6} {item['cohort']:<11} {item['count']:>5,} "
              f"{item['readiness_median']:>17} {item['readiness_p25']:>4} "
              f"{item['readiness_p75']:>4} {item['adult_content_median']:>21}")
print(f"\nReadiness median gap: {summary['readiness_median_gap_before']:+.0f} before, "
      f"{summary['readiness_median_gap_after']:+.0f} after.")


Stage  Audience    Count  Readiness median  P25  P75  Adult-content median
Before 11-17       6,667                50 38.0 59.0                    41
Before Layperson   2,074              35.0 32.0 38.0                  35.0
After  11-17       6,667                38 33.0 49.0                    37
After  Layperson   2,074              56.0 53.0 59.0                  42.0

Readiness median gap: -15 before, +18 after.


## Reallocation matrix

In [3]:
print('Before      After        Count')
for item in summary['transitions']:
    print(f"{item['before']:<11} {item['after']:<11} {item['count']:>5,}")


Before      After        Count
11-17       11-17       4,756
11-17       Layperson   1,911
Layperson   11-17       1,911
Layperson   Layperson     163


## Formats that crossed the boundary

In [4]:
print('Before      After        Format                 Count')
for item in summary['format_transitions']:
    if item['before'] != item['after']:
        print(f"{item['before']:<11} {item['after']:<11} "
              f"{item['format']:<22} {item['count']:>5,}")


Before      After        Format                 Count
11-17       Layperson   book-context             957
11-17       Layperson   missing-word             610
11-17       Layperson   section-context          340
11-17       Layperson   teen-trivia                4
Layperson   11-17       adult-standard         1,911


## Reconciliation and guardrails

In [5]:
assert len(rows) == summary['boundary_question_count'] == 8741
assert len(changes) == summary['changed_question_count'] == 3822
final_counts = {cohort: sum(row['final'] == cohort for row in rows)
                for cohort in ('11-17', 'Layperson')}
assert final_counts == {'11-17': 6667, 'Layperson': 2074}
assert summary['readiness_median_gap_before'] < 0
assert summary['readiness_median_gap_after'] >= 5
assert summary['actual_layperson_to_deacon_median_gap'] >= 5
assert summary['minimum_actual_layperson_category_pool'] >= 20
assert sum(row['before'] == '11-17' and row['final'] == 'Layperson' for row in rows) == 1911
assert sum(row['before'] == 'Layperson' and row['final'] == '11-17' for row in rows) == 1911
print('PASS — counts reconcile, progression is monotonic, and every safeguard holds.')


PASS — counts reconcile, progression is monotonic, and every safeguard holds.


## Interpretation

The original boundary was inverted because passage-inference templates in the teen pool demanded more independent canonical knowledge than most adult Layperson questions. The revised allocation fixes the population ordering while preserving audience sizes, four-choice teen records, safety screens, stable IDs, and all 21 adult gameplay categories. Individual placement remains an editorial estimate until response-rate data is available.